# Body Composition Analysis with Exercise Support
This notebook implements Random Forest Regressor models to predict Body Fat Percentage (BFP), muscle mass changes, and definition scores at different time intervals (3, 6, 9, 12 months) while incorporating exercise data.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
# import matplotlib.pyplot as plt
# import seaborn as sns
from typing import List, Dict
import joblib

## Load and Inspect Data
Load the dataset (`fatLoss.csv`) and display its overview and information.

In [ ]:
# Load the data
try:
    df = pd.read_csv('fatLoss.csv')
    print('Dataset loaded successfully!')
except:
    print('Error loading dataset')
    exit()

print('Dataset Overview:')
print(df.head())
print('\nDataset Info:')
print(df.info())

Dataset loaded successfully!
Dataset Overview:
         exercise_name target_muscle_group      type  age gender  sets  reps  \
0  Barbell Bench Press               Chest  Compound   32      M     4    10   
1  Barbell Bench Press               Chest  Compound   28      F     3    12   
2  Barbell Bench Press               Chest  Compound   39      M     5     8   
3  Barbell Bench Press               Chest  Compound   35      F     4    10   
4  Barbell Bench Press               Chest  Compound   26      M     3    12   

   weight  frequency  protein  calories  sleep    experience  \
0      80          3      150      2800    7.5  Intermediate   
1      45          2      130      2600    7.2      Beginner   
2      90          4      170      3100    8.0      Advanced   
3      50          3      135      2700    7.4  Intermediate   
4      85          2      145      2750    7.6  Intermediate   

   genetic_advantage  actual_fat_loss  daily_deficit  current_weight  height  
0       

## Initial Feature Engineering
Calculate Basal Metabolic Rate (BMR) using the Mifflin-St Jeor Equation and generate initial Body Fat Percentage (BFP) and exercise-related columns if they don't exist.

In [ ]:
# Calculate BMR (Basal Metabolic Rate) using Mifflin-St Jeor Equation
df.loc[df['gender'] == 'M', 'BMR'] = (
    88.362
    + (13.397 * df.loc[df['gender'] == 'M', 'current_weight'])
    + (4.799 * df.loc[df['gender'] == 'M', 'height'])
    - (5.677 * df.loc[df['gender'] == 'M', 'age'])
)

df.loc[df['gender'] == 'F', 'BMR'] = (
    447.593
    + (9.247 * df.loc[df['gender'] == 'F', 'current_weight'])
    + (3.098 * df.loc[df['gender'] == 'F', 'height'])
    - (4.330 * df.loc[df['gender'] == 'F', 'age'])
)

# Generate initial BFP if not present
np.random.seed(42)  # For reproducibility
df['BFP'] = np.random.uniform(5, 50, size=len(df))

# Add exercise-related columns if they don't exist (simulate exercise data for fat loss dataset)
if 'exercise_name' not in df.columns:
    exercise_names = ['Barbell Bench Press', 'Barbell Rows', 'Squats', 'Deadlifts', 'Pull-ups', 
                     'Dumbbell Press', 'Lat Pulldowns', 'Bicep Curls', 'Tricep Extensions', 'Leg Press']
    df['exercise_name'] = np.random.choice(exercise_names, size=len(df))

if 'target_muscle_group' not in df.columns:
    muscle_groups = ['Chest', 'Back', 'Legs', 'Shoulders', 'Arms']
    df['target_muscle_group'] = np.random.choice(muscle_groups, size=len(df))

# Add exercise category if not present
if 'exercise_category' not in df.columns:
    compound_exercises = ['Barbell Bench Press', 'Barbell Rows', 'Squats', 'Deadlifts', 'Pull-ups', 'Lat Pulldowns']
    df['exercise_category'] = df['exercise_name'].apply(
        lambda x: 'Compound' if x in compound_exercises else 'Isolation'
    )

print('BMR, BFP, and exercise data prepared!')

BMR, BFP, and exercise data prepared!


## Feature Engineering
Calculate additional body composition metrics such as BMI, TDEE, estimated fat mass, muscle mass, training intensity, and more.

In [ ]:
# Feature Engineering - Calculate body composition metrics
def calculate_body_composition_features(df):
    """Calculate additional features for body composition prediction"""
    
    # BMI (Body Mass Index)
    df['BMI'] = df['current_weight'] / ((df['height'] / 100) ** 2)
    
    # Activity level (using 5 as default intensity)
    df['activity_level'] = df['frequency'] * 5
    
    # TDEE (Total Daily Energy Expenditure)
    df['TDEE'] = df['BMR'] * (1.2 + (df['activity_level'] * 0.1))
    
    # Estimated fat mass
    df['estimated_fat_mass'] = (df['BFP'] / 100) * df['current_weight']
    
    # Estimate muscle mass
    df['muscle_mass'] = df['current_weight'] - df['estimated_fat_mass']
    
    # Training intensity score
    df['training_intensity'] = (df['weight'] * df['sets'] * df['reps']) / df['current_weight']
    
    # Sleep quality factor
    df['sleep_quality'] = np.where(df['sleep'] >= 7.5, 1, df['sleep'] / 7.5)
    
    # Protein per kg body weight
    df['protein_per_kg'] = df['protein'] / df['current_weight']
    
    # Caloric deficit ratio
    df['deficit_ratio'] = df['daily_deficit'] / df['TDEE']
    
    # Volume calculation
    df['volume'] = df['sets'] * df['reps'] * df['weight']
    
    # Exercise-related features
    df['intensity'] = df['weight'] / np.maximum(df['reps'], 1)
    df['calories_per_kg'] = df['calories'] / np.maximum(df['current_weight'], 1)
    
    return df

# Apply feature engineering
df = calculate_body_composition_features(df)
print('Feature engineering completed!')

Feature engineering completed!


## Encode Categorical Variables
Encode categorical variables like gender, experience, exercise name, muscle group, and exercise category using LabelEncoder.

In [ ]:
# Handle categorical variables
le_gender = LabelEncoder()
le_experience = LabelEncoder()
le_type = LabelEncoder()
le_exercise_name = LabelEncoder()
le_muscle_group = LabelEncoder()
le_exercise_category = LabelEncoder()

df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['experience_encoded'] = le_experience.fit_transform(df['experience'])
df['type_encoded'] = le_type.fit_transform(df['type'])
df['exercise_name_encoded'] = le_exercise_name.fit_transform(df['exercise_name'])
df['muscle_group_encoded'] = le_muscle_group.fit_transform(df['target_muscle_group'])
df['exercise_category_encoded'] = le_exercise_category.fit_transform(df['exercise_category'])

## Generate Target Variables
Generate target variables for BFP, muscle mass, and definition scores at 3, 6, 9, and 12 months.

In [ ]:
# Generate target variables for BFP and muscle mass at different time intervals
np.random.seed(42)

# Calculate predicted weight after fat loss
df['predicted_weight_after_loss'] = df['current_weight'] - df['actual_fat_loss']

# Generate BFP at different months (decreasing over time)
# More aggressive fat loss in early months, slower later
df['bfp_at_3'] = df['BFP'] - np.random.uniform(2, 5, size=len(df))
df['bfp_at_6'] = df['bfp_at_3'] - np.random.uniform(1, 3, size=len(df))
df['bfp_at_9'] = df['bfp_at_6'] - np.random.uniform(0.5, 2, size=len(df))
df['bfp_at_12'] = df['bfp_at_9'] - np.random.uniform(0.5, 1.5, size=len(df))

# Ensure BFP doesn't go below realistic minimum (3% for men, 8% for women)
min_bfp = np.where(df['gender'] == 'M', 3, 8)
df['bfp_at_3'] = np.maximum(df['bfp_at_3'], min_bfp)
df['bfp_at_6'] = np.maximum(df['bfp_at_6'], min_bfp)
df['bfp_at_9'] = np.maximum(df['bfp_at_9'], min_bfp)
df['bfp_at_12'] = np.maximum(df['bfp_at_12'], min_bfp)

# Calculate muscle mass at different time intervals
# Assuming some muscle gain due to training
muscle_gain_factor = 1 + (df['training_intensity'] * 0.001)  # Small muscle gain

df['muscle_mass_at_3'] = (df['predicted_weight_after_loss'] * (1 - df['bfp_at_3']/100)) * muscle_gain_factor
df['muscle_mass_at_6'] = (df['predicted_weight_after_loss'] * (1 - df['bfp_at_6']/100)) * muscle_gain_factor
df['muscle_mass_at_9'] = (df['predicted_weight_after_loss'] * (1 - df['bfp_at_9']/100)) * muscle_gain_factor
df['muscle_mass_at_12'] = (df['predicted_weight_after_loss'] * (1 - df['bfp_at_12']/100)) * muscle_gain_factor

# Calculate definition scores (higher muscle mass + lower BFP = higher definition)
# Scale to 1-10 range
def calculate_definition(muscle_mass, bfp):
    # Higher muscle mass and lower BFP = better definition
    definition_raw = (muscle_mass / 10) * (50 - bfp) / 10
    # Scale to 1-10 range
    return np.clip(definition_raw / definition_raw.max() * 9 + 1, 1, 10)

df['definition_at_3'] = calculate_definition(df['muscle_mass_at_3'], df['bfp_at_3'])
df['definition_at_6'] = calculate_definition(df['muscle_mass_at_6'], df['bfp_at_6'])
df['definition_at_9'] = calculate_definition(df['muscle_mass_at_9'], df['bfp_at_9'])
df['definition_at_12'] = calculate_definition(df['muscle_mass_at_12'], df['bfp_at_12'])

print('Target variables generated for all time intervals!')
print(f'BFP range at 12 months: {df["bfp_at_12"].min():.1f}% - {df["bfp_at_12"].max():.1f}%')
print(f'Definition score range at 12 months: {df["definition_at_12"].min():.1f} - {df["definition_at_12"].max():.1f}')

Target variables generated for all time intervals!
BFP range at 12 months: 3.0% - 41.1%
Definition score range at 12 months: 1.8 - 10.0


## Prepare Feature Matrix
Define the feature set and prepare the feature matrix for training.

In [ ]:
# Define enhanced feature set including exercise features
features = [
    'age', 'gender_encoded', 'current_weight', 'height', 'BMI',
    'sets', 'reps', 'weight', 'frequency', 'training_intensity',
    'protein', 'protein_per_kg', 'calories', 'sleep', 'sleep_quality',
    'experience_encoded', 'genetic_advantage', 'daily_deficit', 'deficit_ratio',
    'type_encoded', 'BFP', 'muscle_mass', 'TDEE', 'activity_level', 
    'exercise_name_encoded', 'muscle_group_encoded', 'exercise_category_encoded',
    'volume', 'intensity', 'calories_per_kg'
]

# Check if all features exist
missing_features = [f for f in features if f not in df.columns]
if missing_features:
    print(f"Missing features: {missing_features}")
else:
    X = df[features]
    print(f"Feature matrix prepared with {len(features)} features")
    print(f"Dataset shape: {X.shape}")

Feature matrix prepared with 30 features
Dataset shape: (390, 30)


## Train Random Forest Models
Train Random Forest Regressor models for BFP, muscle mass, and definition at each time interval (3, 6, 9, 12 months).

In [ ]:
# Train Random Forest models for each time interval and metric
time_intervals = [3, 6, 9, 12]
metrics = ['bfp', 'muscle_mass', 'definition']

# Dictionary to store all models
models = {}
model_scores = {}

# Train models for each combination
for interval in time_intervals:
    models[interval] = {}
    model_scores[interval] = {}
    
    for metric in metrics:
        target_col = f'{metric}_at_{interval}'
        y = df[target_col]
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Train Random Forest model
        rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
        rf.fit(X_train, y_train)
        
        # Make predictions and evaluate
        y_pred = rf.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Store model and scores
        models[interval][metric] = rf
        model_scores[interval][metric] = {'mse': mse, 'r2': r2}
        
        print(f'{metric.upper()} at {interval} months - MSE: {mse:.3f}, R²: {r2:.3f}')

print('\nAll models trained successfully!')

BFP at 3 months - MSE: 0.049, R²: 1.000
MUSCLE_MASS at 3 months - MSE: 1.603, R²: 0.984
DEFINITION at 3 months - MSE: 0.037, R²: 0.992
BFP at 6 months - MSE: 0.526, R²: 0.996
MUSCLE_MASS at 6 months - MSE: 2.061, R²: 0.979
DEFINITION at 6 months - MSE: 0.058, R²: 0.987
BFP at 9 months - MSE: 0.615, R²: 0.995
MUSCLE_MASS at 9 months - MSE: 2.231, R²: 0.977
DEFINITION at 9 months - MSE: 0.087, R²: 0.980
BFP at 12 months - MSE: 0.674, R²: 0.995
MUSCLE_MASS at 12 months - MSE: 2.262, R²: 0.977
DEFINITION at 12 months - MSE: 0.107, R²: 0.975

All models trained successfully!


## Prediction Function
Define a function to predict body composition journey for a new user with multiple exercises.

In [ ]:
# Function to safely encode new data
def safe_transform(encoder, value, default_value=0):
    """Safely transform a value using a label encoder, returning default if not found"""
    try:
        if value in encoder.classes_:
            return encoder.transform([value])[0]
        else:
            print(f"Warning: '{value}' not found in training data, using default")
            return default_value
    except:
        return default_value

# Enhanced prediction function for multiple exercises
def predict_body_composition_journey_with_exercises(age: int, gender: str, exercises: List[Dict], 
                                                  frequency: int, protein: float, calories: int, 
                                                  sleep: float, experience: str, genetic_advantage: int = 3,
                                                  daily_deficit: float = 500, initial_bfp: float = 20.0,
                                                  current_weight: float = 80.0, height: float = 180.0):
    """
    Predict complete body composition journey for a new user with multiple exercises
    Returns predictions for BFP, muscle mass, and definition at 3, 6, 9, 12 months
    """
    
    try:
        # Define difficulty weights based on exercise category
        difficulty_weights = {
            'Compound': 1.0,
            'Isolation': 0.8
        }
        
        # Calculate total volume and get primary exercise characteristics
        total_volume = 0
        primary_exercise = None
        max_volume = 0
        
        for ex in exercises:
            volume = ex['sets'] * ex['reps'] * ex['weight'] * difficulty_weights.get(ex.get('exercise_category', 'Compound'), 1.0)
            total_volume += volume
            if volume > max_volume:
                max_volume = volume
                primary_exercise = ex
        
        # Convert total volume to equivalent sets, reps, weight
        equiv_sets = min(10, primary_exercise['sets'])
        equiv_reps = min(20, primary_exercise['reps'])
        equiv_weight = min(300, total_volume / (equiv_sets * equiv_reps))
        
        # Calculate BMR
        if gender == 'M':
            bmr = (88.362 + (13.397 * current_weight) + 
                   (4.799 * height) - (5.677 * age))
        else:
            bmr = (447.593 + (9.247 * current_weight) + 
                   (3.098 * height) - (4.330 * age))
        
        # Calculate derived features
        bmi = current_weight / ((height / 100) ** 2)
        activity_level = frequency * 5
        tdee = bmr * (1.2 + (activity_level * 0.1))
        estimated_fat_mass = (initial_bfp / 100) * current_weight
        muscle_mass = current_weight - estimated_fat_mass
        training_intensity = (equiv_weight * equiv_sets * equiv_reps) / current_weight
        sleep_quality = 1 if sleep >= 7.5 else sleep / 7.5
        protein_per_kg = protein / current_weight
        deficit_ratio = daily_deficit / tdee
        volume = equiv_sets * equiv_reps * equiv_weight
        intensity = equiv_weight / max(equiv_reps, 1)
        calories_per_kg = calories / max(current_weight, 1)
        
        # Encode categorical variables
        gender_encoded = safe_transform(le_gender, gender)
        experience_encoded = safe_transform(le_experience, experience)
        exercise_name_encoded = safe_transform(le_exercise_name, primary_exercise['exercise_name'])
        muscle_group_encoded = safe_transform(le_muscle_group, primary_exercise.get('target_muscle_group', 'Chest'))
        exercise_category_encoded = safe_transform(le_exercise_category, primary_exercise.get('exercise_category', 'Compound'))
        
        # Use default for 'type' since it's not provided in the exercise data
        type_encoded = 0  # Default value
        
        # Create input array
        user_data = np.array([[
            age, gender_encoded, current_weight, height, bmi,
            equiv_sets, equiv_reps, equiv_weight, frequency, training_intensity,
            protein, protein_per_kg, calories, sleep, sleep_quality,
            experience_encoded, genetic_advantage, daily_deficit, deficit_ratio,
            type_encoded, initial_bfp, muscle_mass, tdee, activity_level,
            exercise_name_encoded, muscle_group_encoded, exercise_category_encoded,
            volume, intensity, calories_per_kg
        ]])
        
        # Make predictions for all time intervals
        predictions = {}
        
        for interval in time_intervals:
            predictions[interval] = {}
            
            for metric in metrics:
                pred = models[interval][metric].predict(user_data)[0]
                predictions[interval][metric] = max(pred, 0)  # Ensure non-negative values
        
        return predictions
        
    except Exception as e:
        print(f"Error in prediction: {str(e)}")
        return None

## Test Prediction Function
Test the prediction function with an example set of exercises and user data.

In [ ]:
# Example usage with multiple exercises
exercises = [
    {
        'exercise_name': 'Barbell Bench Press',
        'sets': 4,
        'reps': 10,
        'weight': 80,
        'target_muscle_group': 'Chest',
        'exercise_category': 'Compound'
    },
    {
        'exercise_name': 'Barbell Rows',
        'sets': 4,
        'reps': 10,
        'weight': 75,
        'target_muscle_group': 'Back',
        'exercise_category': 'Compound'
    },
    {
        'exercise_name': 'Squats',
        'sets': 4,
        'reps': 12,
        'weight': 100,
        'target_muscle_group': 'Legs',
        'exercise_category': 'Compound'
    }
]

# Test the enhanced prediction function
journey = predict_body_composition_journey_with_exercises(
    age=28,
    gender='M',
    exercises=exercises,
    frequency=4,
    protein=150,
    calories=2200,
    sleep=8.0,
    experience='Intermediate',
    genetic_advantage=4,
    daily_deficit=500,
    initial_bfp=18.0,
    current_weight=85.0,
    height=180.0
)

# Display results
if journey:
    print('\n=== BODY COMPOSITION JOURNEY PREDICTION WITH EXERCISES ===')
    print(f'Starting Stats: Weight: 85kg, BFP: 18.0%')
    print(f'Workout: {len(exercises)} exercises with total volume focus')
    print('\nPredicted Progress:')

    for interval in time_intervals:
        print(f'\nMonth {interval}:')
        print(f'  Body Fat: {journey[interval]["bfp"]:.1f}%')
        print(f'  Muscle Mass: {journey[interval]["muscle_mass"]:.1f}kg')
        print(f'  Definition Score: {journey[interval]["definition"]:.1f}/10')


=== BODY COMPOSITION JOURNEY PREDICTION WITH EXERCISES ===
Starting Stats: Weight: 85kg, BFP: 18.0%
Workout: 3 exercises with total volume focus

Predicted Progress:

Month 3:
  Body Fat: 15.2%
  Muscle Mass: 61.0kg
  Definition Score: 6.5/10

Month 6:
  Body Fat: 13.5%
  Muscle Mass: 62.5kg
  Definition Score: 6.9/10

Month 9:
  Body Fat: 12.0%
  Muscle Mass: 63.7kg
  Definition Score: 7.2/10

Month 12:
  Body Fat: 11.1%
  Muscle Mass: 64.2kg
  Definition Score: 7.2/10


c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2

## Save Models and Encoders
Save the trained models and label encoders for future use.

In [ ]:
# Save all models and encoders
joblib.dump(models, 'body_composition_models.pkl')
joblib.dump(le_gender, 'le_gender_body.pkl')
joblib.dump(le_experience, 'le_experience_body.pkl')
joblib.dump(le_exercise_name, 'le_exercise_name_body.pkl')
joblib.dump(le_muscle_group, 'le_muscle_group_body.pkl')
joblib.dump(le_exercise_category, 'le_exercise_category_body.pkl')

print('\nModels and encoders saved successfully!')
print(f'Available exercises: {len(le_exercise_name.classes_)}')
print(f'Available muscle groups: {len(le_muscle_group.classes_)}')


Models and encoders saved successfully!
Available exercises: 39
Available muscle groups: 4


In [ ]:
len(df['exercise_name'].unique())

39

In [ ]:
df.columns

Index(['exercise_name', 'target_muscle_group', 'type', 'age', 'gender', 'sets',
       'reps', 'weight', 'frequency', 'protein', 'calories', 'sleep',
       'experience', 'genetic_advantage', 'actual_fat_loss', 'daily_deficit',
       'current_weight', 'height', 'BMR', 'BFP', 'exercise_category', 'BMI',
       'activity_level', 'TDEE', 'estimated_fat_mass', 'muscle_mass',
       'training_intensity', 'sleep_quality', 'protein_per_kg',
       'deficit_ratio', 'volume', 'intensity', 'calories_per_kg',
       'gender_encoded', 'experience_encoded', 'type_encoded',
       'exercise_name_encoded', 'muscle_group_encoded',
       'exercise_category_encoded', 'predicted_weight_after_loss', 'bfp_at_3',
       'bfp_at_6', 'bfp_at_9', 'bfp_at_12', 'muscle_mass_at_3',
       'muscle_mass_at_6', 'muscle_mass_at_9', 'muscle_mass_at_12',
       'definition_at_3', 'definition_at_6', 'definition_at_9',
       'definition_at_12'],
      dtype='object')

## Generate Muscle Group Definition Scores
Generate definition scores for each muscle group (Arms, Chest, Back, Quads) at 3, 6, 9, and 12 month milestones.

In [ ]:
# Generate muscle group-specific definition scores for each time interval
muscle_groups = ['Arms', 'Chest', 'Back', 'Quads']
time_intervals = [3, 6, 9, 12]

# Function to calculate muscle group specific definition scores
def calculate_muscle_group_definition(base_definition, muscle_mass, bfp, muscle_group, target_muscle_group):
    """
    Calculate definition score for specific muscle group based on:
    - Base definition score
    - Whether this muscle group is the target of the workout
    - Muscle mass and body fat percentage
    """
    # Base definition score (0-10 range)
    group_definition = base_definition.copy()
    
    # Add bonus if this muscle group matches the target muscle group
    target_bonus = np.where(target_muscle_group == muscle_group, 0.5, 0)
    
    # Add variation based on muscle group characteristics
    muscle_multipliers = {
        'Arms': 1.0,     # Standard definition
        'Chest': 1.1,    # Slightly easier to define
        'Back': 0.9,     # Harder to see definition
        'Quads': 1.05    # Good visibility for definition
    }
    
    # Apply muscle-specific multiplier and target bonus
    group_definition = group_definition * muscle_multipliers[muscle_group] + target_bonus
    
    # Add some random variation (±0.5)
    variation = np.random.uniform(-0.5, 0.5, size=len(group_definition))
    group_definition += variation
    
    # Ensure values stay within 0-10 range
    group_definition = np.clip(group_definition, 0, 10)
    
    return group_definition

# Generate definition scores for each muscle group at each time interval
np.random.seed(42)  # For reproducibility

for interval in time_intervals:
    base_definition_col = f'definition_at_{interval}'
    muscle_mass_col = f'muscle_mass_at_{interval}'
    bfp_col = f'bfp_at_{interval}'
    
    for muscle_group in muscle_groups:
        col_name = f'{muscle_group.lower()}_definition_at_{interval}'
        
        df[col_name] = calculate_muscle_group_definition(
            df[base_definition_col],
            df[muscle_mass_col], 
            df[bfp_col],
            muscle_group,
            df['target_muscle_group']
        )

print('Muscle group definition scores generated for all time intervals!')

# Display sample of the new columns
sample_cols = ['target_muscle_group', 'definition_at_3', 'arms_definition_at_3', 
               'chest_definition_at_3', 'back_definition_at_3', 'quads_definition_at_3']
print('\nSample of muscle group definition scores:')
print(df[sample_cols].head())

# Show statistics for 12-month predictions
print('\n12-Month Muscle Group Definition Statistics:')
for muscle_group in muscle_groups:
    col_name = f'{muscle_group.lower()}_definition_at_12'
    print(f'{muscle_group}: {df[col_name].min():.1f} - {df[col_name].max():.1f} (avg: {df[col_name].mean():.1f})')

Muscle group definition scores generated for all time intervals!

Sample of muscle group definition scores:
  target_muscle_group  definition_at_3  arms_definition_at_3  \
0               Chest         5.654645              5.529186   
1               Chest         1.595796              2.046511   
2               Chest         2.970718              3.202712   
3               Chest         3.304619              3.403278   
4               Chest         7.602523              7.258542   

   chest_definition_at_3  back_definition_at_3  quads_definition_at_3  
0               7.210615              5.283129               6.054305  
1               2.167994              1.478941               2.156772  
2               3.639808              2.425445               3.251067  
3               4.411494              2.819853               3.229654  
4               8.703579              6.523869               8.116655  

12-Month Muscle Group Definition Statistics:
Arms: 1.6 - 10.0 (avg: 5.1)
C

## Train Muscle Group Definition Models
Train Random Forest models to predict definition scores for each muscle group at different time intervals.

In [ ]:
# Train Random Forest models for muscle group definition scores
muscle_group_models = {}
muscle_group_scores = {}

# Extended metrics to include muscle group definitions
extended_metrics = ['bfp', 'muscle_mass', 'definition'] + [f'{mg.lower()}_definition' for mg in muscle_groups]

print('Training muscle group definition models...')

for interval in time_intervals:
    muscle_group_models[interval] = {}
    muscle_group_scores[interval] = {}
    
    # Train models for each muscle group definition
    for muscle_group in muscle_groups:
        metric_name = f'{muscle_group.lower()}_definition'
        target_col = f'{muscle_group.lower()}_definition_at_{interval}'
        
        if target_col in df.columns:
            y = df[target_col]
            
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            # Train Random Forest model
            rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
            rf.fit(X_train, y_train)
            
            # Make predictions and evaluate
            y_pred = rf.predict(X_test)
            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            # Store model and scores
            muscle_group_models[interval][metric_name] = rf
            muscle_group_scores[interval][metric_name] = {'mse': mse, 'r2': r2}
            
            print(f'{muscle_group} definition at {interval} months - MSE: {mse:.3f}, R²: {r2:.3f}')

# Combine all models into one dictionary
all_models = {}
for interval in time_intervals:
    all_models[interval] = {**models[interval], **muscle_group_models[interval]}

print('\nAll muscle group definition models trained successfully!')
print(f'Total models per interval: {len(all_models[3])} (original: {len(models[3])}, muscle groups: {len(muscle_group_models[3])})')

Training muscle group definition models...
Arms definition at 3 months - MSE: 0.038, R²: 0.989
Chest definition at 3 months - MSE: 0.193, R²: 0.965
Back definition at 3 months - MSE: 0.157, R²: 0.959
Quads definition at 3 months - MSE: 0.161, R²: 0.969
Arms definition at 6 months - MSE: 0.172, R²: 0.960
Chest definition at 6 months - MSE: 0.206, R²: 0.962
Back definition at 6 months - MSE: 0.157, R²: 0.958
Quads definition at 6 months - MSE: 0.144, R²: 0.970
Arms definition at 9 months - MSE: 0.174, R²: 0.962
Chest definition at 9 months - MSE: 0.238, R²: 0.955
Back definition at 9 months - MSE: 0.195, R²: 0.948
Quads definition at 9 months - MSE: 0.176, R²: 0.964
Arms definition at 12 months - MSE: 0.167, R²: 0.963
Chest definition at 12 months - MSE: 0.235, R²: 0.955
Back definition at 12 months - MSE: 0.151, R²: 0.960
Quads definition at 12 months - MSE: 0.119, R²: 0.974

All muscle group definition models trained successfully!
Total models per interval: 7 (original: 3, muscle group

## Enhanced Prediction Function with Muscle Groups
Update the prediction function to include muscle group definition scores.

In [ ]:
# Enhanced prediction function with muscle group definitions
def predict_body_composition_with_muscle_groups(age: int, gender: str, exercises: List[Dict], 
                                               frequency: int, protein: float, calories: int, 
                                               sleep: float, experience: str, genetic_advantage: int = 3,
                                               daily_deficit: float = 500, initial_bfp: float = 20.0,
                                               current_weight: float = 80.0, height: float = 180.0):
    """
    Enhanced prediction function that includes muscle group definition scores
    Returns predictions for BFP, muscle mass, overall definition, and muscle group definitions at 3, 6, 9, 12 months
    """
    
    try:
        # Use the same feature calculation as the original function
        difficulty_weights = {
            'Compound': 1.0,
            'Isolation': 0.8
        }
        
        # Calculate total volume and get primary exercise characteristics
        total_volume = 0
        primary_exercise = None
        max_volume = 0
        
        for ex in exercises:
            volume = ex['sets'] * ex['reps'] * ex['weight'] * difficulty_weights.get(ex.get('exercise_category', 'Compound'), 1.0)
            total_volume += volume
            if volume > max_volume:
                max_volume = volume
                primary_exercise = ex
        
        # Convert total volume to equivalent sets, reps, weight
        equiv_sets = min(10, primary_exercise['sets'])
        equiv_reps = min(20, primary_exercise['reps'])
        equiv_weight = min(300, total_volume / (equiv_sets * equiv_reps))
        
        # Calculate BMR
        if gender == 'M':
            bmr = (88.362 + (13.397 * current_weight) + 
                   (4.799 * height) - (5.677 * age))
        else:
            bmr = (447.593 + (9.247 * current_weight) + 
                   (3.098 * height) - (4.330 * age))
        
        # Calculate derived features
        bmi = current_weight / ((height / 100) ** 2)
        activity_level = frequency * 5
        tdee = bmr * (1.2 + (activity_level * 0.1))
        estimated_fat_mass = (initial_bfp / 100) * current_weight
        muscle_mass = current_weight - estimated_fat_mass
        training_intensity = (equiv_weight * equiv_sets * equiv_reps) / current_weight
        sleep_quality = 1 if sleep >= 7.5 else sleep / 7.5
        protein_per_kg = protein / current_weight
        deficit_ratio = daily_deficit / tdee
        volume = equiv_sets * equiv_reps * equiv_weight
        intensity = equiv_weight / max(equiv_reps, 1)
        calories_per_kg = calories / max(current_weight, 1)
        
        # Encode categorical variables
        gender_encoded = safe_transform(le_gender, gender)
        experience_encoded = safe_transform(le_experience, experience)
        exercise_name_encoded = safe_transform(le_exercise_name, primary_exercise['exercise_name'])
        muscle_group_encoded = safe_transform(le_muscle_group, primary_exercise.get('target_muscle_group', 'Chest'))
        exercise_category_encoded = safe_transform(le_exercise_category, primary_exercise.get('exercise_category', 'Compound'))
        
        # Use default for 'type' since it's not provided in the exercise data
        type_encoded = 0  # Default value
        
        # Create input array
        user_data = np.array([[
            age, gender_encoded, current_weight, height, bmi,
            equiv_sets, equiv_reps, equiv_weight, frequency, training_intensity,
            protein, protein_per_kg, calories, sleep, sleep_quality,
            experience_encoded, genetic_advantage, daily_deficit, deficit_ratio,
            type_encoded, initial_bfp, muscle_mass, tdee, activity_level,
            exercise_name_encoded, muscle_group_encoded, exercise_category_encoded,
            volume, intensity, calories_per_kg
        ]])
        
        # Make predictions for all time intervals and metrics
        predictions = {}
        
        for interval in time_intervals:
            predictions[interval] = {}
            
            # Predict original metrics
            for metric in ['bfp', 'muscle_mass', 'definition']:
                pred = all_models[interval][metric].predict(user_data)[0]
                predictions[interval][metric] = max(pred, 0)  # Ensure non-negative values
            
            # Predict muscle group definitions
            muscle_group_definitions = {}
            for muscle_group in muscle_groups:
                metric_name = f'{muscle_group.lower()}_definition'
                if metric_name in all_models[interval]:
                    pred = all_models[interval][metric_name].predict(user_data)[0]
                    muscle_group_definitions[muscle_group.lower()] = max(0, min(10, pred))  # Clamp to 0-10
            
            predictions[interval]['muscle_groups'] = muscle_group_definitions
        
        return predictions
        
    except Exception as e:
        print(f"Error in enhanced prediction: {str(e)}")
        return None

# Test the enhanced prediction function
print('Testing enhanced prediction function with muscle groups...')

test_exercises = [
    {
        'exercise_name': 'Barbell Bench Press',
        'sets': 4,
        'reps': 10,
        'weight': 80,
        'target_muscle_group': 'Chest',
        'exercise_category': 'Compound'
    },
    {
        'exercise_name': 'Bicep Curls',
        'sets': 3,
        'reps': 12,
        'weight': 25,
        'target_muscle_group': 'Arms',
        'exercise_category': 'Isolation'
    }
]

enhanced_journey = predict_body_composition_with_muscle_groups(
    age=28,
    gender='M',
    exercises=test_exercises,
    frequency=4,
    protein=150,
    calories=2200,
    sleep=8.0,
    experience='Intermediate',
    genetic_advantage=4,
    daily_deficit=500,
    initial_bfp=18.0,
    current_weight=85.0,
    height=180.0
)

# Display enhanced results
if enhanced_journey:
    print('\n=== ENHANCED BODY COMPOSITION JOURNEY WITH MUSCLE GROUPS ===')
    print(f'Starting Stats: Weight: 85kg, BFP: 18.0%')
    print(f'Primary Exercise: Chest-focused workout')
    
    for interval in time_intervals:
        print(f'\nMonth {interval}:')
        print(f'  Body Fat: {enhanced_journey[interval]["bfp"]:.1f}%')
        print(f'  Muscle Mass: {enhanced_journey[interval]["muscle_mass"]:.1f}kg')
        print(f'  Overall Definition: {enhanced_journey[interval]["definition"]:.1f}/10')
        print('  Muscle Group Definitions:')
        for muscle, score in enhanced_journey[interval]['muscle_groups'].items():
            print(f'    {muscle.capitalize()}: {score:.1f}/10')

Testing enhanced prediction function with muscle groups...


c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2


=== ENHANCED BODY COMPOSITION JOURNEY WITH MUSCLE GROUPS ===
Starting Stats: Weight: 85kg, BFP: 18.0%
Primary Exercise: Chest-focused workout

Month 3:
  Body Fat: 15.2%
  Muscle Mass: 60.9kg
  Overall Definition: 6.5/10
  Muscle Group Definitions:
    Arms: 6.3/10
    Chest: 7.4/10
    Back: 5.8/10
    Quads: 6.7/10

Month 6:
  Body Fat: 13.7%
  Muscle Mass: 62.4kg
  Overall Definition: 6.9/10
  Muscle Group Definitions:
    Arms: 7.0/10
    Chest: 8.1/10
    Back: 6.4/10
    Quads: 7.5/10

Month 9:
  Body Fat: 12.3%
  Muscle Mass: 63.7kg
  Overall Definition: 7.2/10
  Muscle Group Definitions:
    Arms: 7.2/10
    Chest: 8.3/10
    Back: 6.5/10
    Quads: 7.4/10

Month 12:
  Body Fat: 11.1%
  Muscle Mass: 64.4kg
  Overall Definition: 7.1/10
  Muscle Group Definitions:
    Arms: 7.5/10
    Chest: 8.2/10
    Back: 6.3/10
    Quads: 7.5/10


c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


## Save Enhanced Models
Save the enhanced models that include muscle group definition predictions.

In [ ]:
# Save enhanced models that include muscle group definitions
joblib.dump(all_models, 'enhanced_body_composition_models.pkl')

print('Enhanced models saved successfully!')
print(f'Models saved for {len(time_intervals)} time intervals')
print(f'Each interval includes: {len(all_models[3])} total models')
print('- 3 original metrics: bfp, muscle_mass, definition')
print(f'- {len(muscle_groups)} muscle group definitions: {[f"{mg.lower()}_definition" for mg in muscle_groups]}')

# Verify what's in the saved models
print('\nModel structure verification:')
for interval in [3, 12]:  # Show first and last intervals
    print(f'Month {interval} models: {list(all_models[interval].keys())}')

Enhanced models saved successfully!
Models saved for 4 time intervals
Each interval includes: 7 total models
- 3 original metrics: bfp, muscle_mass, definition
- 4 muscle group definitions: ['arms_definition', 'chest_definition', 'back_definition', 'quads_definition']

Model structure verification:
Month 3 models: ['bfp', 'muscle_mass', 'definition', 'arms_definition', 'chest_definition', 'back_definition', 'quads_definition']
Month 12 models: ['bfp', 'muscle_mass', 'definition', 'arms_definition', 'chest_definition', 'back_definition', 'quads_definition']


In [19]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import numpy as np
from typing import List, Dict

app = Flask(__name__)
CORS(app)  # Enable CORS for all routes

# Load enhanced models and encoders
try:
    all_models = joblib.load('enhanced_body_composition_models.pkl')
    le_gender = joblib.load('le_gender_body.pkl')
    le_experience = joblib.load('le_experience_body.pkl')
    le_exercise_name = joblib.load('le_exercise_name_body.pkl')
    le_muscle_group = joblib.load('le_muscle_group_body.pkl')
    le_exercise_category = joblib.load('le_exercise_category_body.pkl')
except Exception as e:
    print(f"Error loading models or encoders: {str(e)}")
    exit()

# Define time intervals, metrics, and muscle groups as in the notebook
time_intervals = [3, 6, 9, 12]
metrics = ['bfp', 'muscle_mass', 'definition']
muscle_groups = ['Arms', 'Chest', 'Back', 'Quads']

# Function to safely encode new data (from notebook)
def safe_transform(encoder, value, default_value=0):
    """Safely transform a value using a label encoder, returning default if not found"""
    try:
        if value in encoder.classes_:
            return encoder.transform([value])[0]
        else:
            print(f"Warning: '{value}' not found in training data, using default")
            return default_value
    except:
        return default_value

# Enhanced prediction function with muscle group definitions
def predict_body_composition_with_muscle_groups(age: int, gender: str, exercises: List[Dict], 
                                               frequency: int, protein: float, calories: int, 
                                               sleep: float, experience: str, genetic_advantage: int = 3,
                                               daily_deficit: float = 500, initial_bfp: float = 20.0,
                                               current_weight: float = 80.0, height: float = 180.0):
    """
    Enhanced prediction function that includes muscle group definition scores
    Returns predictions for BFP, muscle mass, overall definition, and muscle group definitions at 3, 6, 9, 12 months
    """
    
    try:
        # Use the same feature calculation as the original function
        difficulty_weights = {
            'Compound': 1.0,
            'Isolation': 0.8
        }
        
        # Calculate total volume and get primary exercise characteristics
        total_volume = 0
        primary_exercise = None
        max_volume = 0
        
        for ex in exercises:
            volume = ex['sets'] * ex['reps'] * ex['weight'] * difficulty_weights.get(ex.get('exercise_category', 'Compound'), 1.0)
            total_volume += volume
            if volume > max_volume:
                max_volume = volume
                primary_exercise = ex
        
        # Convert total volume to equivalent sets, reps, weight
        equiv_sets = min(10, primary_exercise['sets'])
        equiv_reps = min(20, primary_exercise['reps'])
        equiv_weight = min(300, total_volume / (equiv_sets * equiv_reps))
        
        # Calculate BMR
        if gender == 'M':
            bmr = (88.362 + (13.397 * current_weight) + 
                   (4.799 * height) - (5.677 * age))
        else:
            bmr = (447.593 + (9.247 * current_weight) + 
                   (3.098 * height) - (4.330 * age))
        
        # Calculate derived features
        bmi = current_weight / ((height / 100) ** 2)
        activity_level = frequency * 5
        tdee = bmr * (1.2 + (activity_level * 0.1))
        estimated_fat_mass = (initial_bfp / 100) * current_weight
        muscle_mass = current_weight - estimated_fat_mass
        training_intensity = (equiv_weight * equiv_sets * equiv_reps) / current_weight
        sleep_quality = 1 if sleep >= 7.5 else sleep / 7.5
        protein_per_kg = protein / current_weight
        deficit_ratio = daily_deficit / tdee
        volume = equiv_sets * equiv_reps * equiv_weight
        intensity = equiv_weight / max(equiv_reps, 1)
        calories_per_kg = calories / max(current_weight, 1)
        
        # Encode categorical variables
        gender_encoded = safe_transform(le_gender, gender)
        experience_encoded = safe_transform(le_experience, experience)
        exercise_name_encoded = safe_transform(le_exercise_name, primary_exercise['exercise_name'])
        muscle_group_encoded = safe_transform(le_muscle_group, primary_exercise.get('target_muscle_group', 'Chest'))
        exercise_category_encoded = safe_transform(le_exercise_category, primary_exercise.get('exercise_category', 'Compound'))
        
        # Use default for 'type' since it's not provided in the exercise data
        type_encoded = 0  # Default value
        
        # Create input array
        user_data = np.array([[
            age, gender_encoded, current_weight, height, bmi,
            equiv_sets, equiv_reps, equiv_weight, frequency, training_intensity,
            protein, protein_per_kg, calories, sleep, sleep_quality,
            experience_encoded, genetic_advantage, daily_deficit, deficit_ratio,
            type_encoded, initial_bfp, muscle_mass, tdee, activity_level,
            exercise_name_encoded, muscle_group_encoded, exercise_category_encoded,
            volume, intensity, calories_per_kg
        ]])
        
        # Make predictions for all time intervals and metrics
        predictions = {}
        
        for interval in time_intervals:
            predictions[interval] = {}
            
            # Predict original metrics
            for metric in ['bfp', 'muscle_mass', 'definition']:
                pred = all_models[interval][metric].predict(user_data)[0]
                predictions[interval][metric] = max(pred, 0)  # Ensure non-negative values
            
            # Predict muscle group definitions
            muscle_group_definitions = {}
            for muscle_group in muscle_groups:
                metric_name = f'{muscle_group.lower()}_definition'
                if metric_name in all_models[interval]:
                    pred = all_models[interval][metric_name].predict(user_data)[0]
                    muscle_group_definitions[muscle_group.lower()] = max(0, min(10, pred))  # Clamp to 0-10
            
            predictions[interval]['muscle_groups'] = muscle_group_definitions
        
        return predictions
        
    except Exception as e:
        print(f"Error in enhanced prediction: {str(e)}")
        return None

@app.route('/cut', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        age = data.get('age')
        gender = data.get('gender')
        exercises = data.get('exercises')
        frequency = data.get('frequency')
        protein = data.get('protein')
        calories = data.get('calories')
        sleep = data.get('sleep')
        experience = data.get('experience')
        genetic_advantage = data.get('genetic_advantage', 3)
        daily_deficit = data.get('daily_deficit', 500)
        initial_bfp = data.get('initial_bfp', 20.0)
        current_weight = data.get('current_weight', 80.0)
        height = data.get('height', 180.0)

        # Validate required inputs
        if not all([age, gender, exercises, frequency, protein, calories, sleep, experience]):
            return jsonify({'error': 'Missing required parameters'}), 400

        prediction = predict_body_composition_with_muscle_groups(
            age=age,
            gender=gender,
            exercises=exercises,
            frequency=frequency,
            protein=protein,
            calories=calories,
            sleep=sleep,
            experience=experience,
            genetic_advantage=genetic_advantage,
            daily_deficit=daily_deficit,
            initial_bfp=initial_bfp,
            current_weight=current_weight,
            height=height
        )

        if prediction is None:
            return jsonify({'error': 'Prediction failed'}), 500

        return jsonify({'result': prediction})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    app.run(host='localhost', port=3002)

 * Serving Flask app '__main__'
 * Debug mode: off
 * Debug mode: off


 * Running on http://localhost:3002
Press CTRL+C to quit

 * Running on http://localhost:3002
Press CTRL+C to quit
127.0.0.1 - - [08/Sep/2025 18:00:38] "OPTIONS /cut HTTP/1.1" 200 -
127.0.0.1 - - [08/Sep/2025 18:00:38] "OPTIONS /cut HTTP/1.1" 200 -
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(